In [0]:
from pyspark.sql import functions as F

events = spark.table(
    "fintech_lakehouse.silver.order_events_clean_dev"
)

orders = spark.table(
    "fintech_lakehouse.silver.orders_current_dev"
)

quarantine = spark.table(
    "fintech_lakehouse.silver.order_events_quarantine_dev"
)

# These expected values apply to our verified 72-event batch.
assert events.count() == 72, "Expected 72 clean events."
assert orders.count() == 30, "Expected 30 current orders."
assert quarantine.count() == 0, "Expected no rejected events."

assert events.select("event_id").distinct().count() == 72, (
    "Duplicate event IDs found."
)

assert orders.select("order_id").distinct().count() == 30, (
    "Duplicate current order IDs found."
)

expected_statuses = {
    "CREATED": 6,
    "PAID": 6,
    "SHIPPED": 12,
    "CANCELLED": 6,
}

actual_statuses = {
    row["status"]: row["count"]
    for row in orders.groupBy("status").count().collect()
}

assert actual_statuses == expected_statuses, (
    f"Unexpected status counts: {actual_statuses}"
)

# Check that every order holds its highest available sequence.
expected_sequences = events.groupBy("order_id").agg(
    F.max("sequence_number").alias("expected_sequence")
)

sequence_errors = (
    orders.join(expected_sequences, "order_id", "full")
    .filter(
        ~F.col("sequence_number").eqNullSafe(
            F.col("expected_sequence")
        )
    )
)

assert sequence_errors.count() == 0, (
    "Current orders do not match the latest event sequences."
)

# Independently calculate creation time from INSERT events.
expected_creation = (
    events.filter(F.col("event_type") == "INSERT")
    .groupBy("order_id")
    .agg(F.min("event_timestamp").alias("expected_created_at"))
)

creation_errors = (
    orders.join(expected_creation, "order_id", "full")
    .filter(
        F.col("order_created_at").isNull()
        | F.col("expected_created_at").isNull()
        | ~F.col("order_created_at").eqNullSafe(
            F.col("expected_created_at")
        )
    )
)

assert creation_errors.count() == 0, (
    "Order creation timestamps are missing or incorrect."
)

assert orders.filter(
    ~F.col("is_cancelled").eqNullSafe(
        F.col("status") == "CANCELLED"
    )
).count() == 0, "Cancellation flags do not match statuses."

print("PASS: All 9 baseline Silver checks passed.")